In [2]:
from pathlib import Path
import subprocess, os

REPO_HINT = Path("/Users/mkrishan/Documents/boolearn-main")  # set None for public release

def find_repo_root():
    candidates = []
    if REPO_HINT is not None:
        candidates.append(Path(REPO_HINT).expanduser().resolve())
    cwd = Path.cwd().resolve()
    candidates += [cwd] + list(cwd.parents)
    home = Path.home()
    candidates += [home/"boolearn-main", home/"Documents"/"boolearn-main", home/"Desktop"/"boolearn-main"]
    for p in candidates:
        if (p/"boolearn"/"src"/"makefile").exists():
            return p
    raise RuntimeError(f"Could not locate repo root from {cwd}. Set REPO_HINT.")

ROOT = find_repo_root()
SRC  = ROOT / "boolearn" / "src"
DATA = ROOT / "data"
RESULTS = ROOT / "results"
RESULTS.mkdir(exist_ok=True)

print("✅ ROOT:", ROOT)
print("✅ SRC:", SRC)
print("✅ DATA:", DATA)
print("✅ RESULTS:", RESULTS)

subprocess.run(["make","all"], cwd=str(SRC), check=True)
os.environ["PATH"] = f"{SRC}:{os.environ.get('PATH','')}"
print("✅ Built + PATH updated")

✅ ROOT: /Users/mkrishan/Documents/boolearn-main
✅ SRC: /Users/mkrishan/Documents/boolearn-main/boolearn/src
✅ DATA: /Users/mkrishan/Documents/boolearn-main/data
✅ RESULTS: /Users/mkrishan/Documents/boolearn-main/results
make: Nothing to be done for `all'.
✅ Built + PATH updated


In [5]:
from pathlib import Path
import subprocess

train   = SRC / "train"
layered = SRC / "layered"
assert train.exists(), train
assert layered.exists(), layered

dat = DATA / "autocode" / "auto16.dat"
assert dat.exists(), dat

OUT = RESULTS / "02_autoencoder_auto16"
NET_DIR = OUT / "nets"
RUN_DIR = OUT / "runs"
NET_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=True)

print("✅ train:", train)
print("✅ layered:", layered)
print("✅ dat:", dat)
print("✅ NET_DIR:", NET_DIR)
print("✅ RUN_DIR:", RUN_DIR)

✅ train: /Users/mkrishan/Documents/boolearn-main/boolearn/src/train
✅ layered: /Users/mkrishan/Documents/boolearn-main/boolearn/src/layered
✅ dat: /Users/mkrishan/Documents/boolearn-main/data/autocode/auto16.dat
✅ NET_DIR: /Users/mkrishan/Documents/boolearn-main/results/02_autoencoder_auto16/nets
✅ RUN_DIR: /Users/mkrishan/Documents/boolearn-main/results/02_autoencoder_auto16/runs


In [7]:
import subprocess

wth = NET_DIR / "auto16.wth"
net = NET_DIR / "auto16.net"

# Architecture from the guide:
# 4
# 16 16 4 16 16
with open(wth, "w") as f:
    f.write("4\n16 16 4 16 16\n")

print("Wrote:", wth)
print(wth.read_text())

if not net.exists():
    cmd = [str(layered), "0.5", str(wth), str(net)]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, cwd=str(NET_DIR), check=False)

assert net.exists() and net.stat().st_size > 0
print("✅ net:", net, "| bytes:", net.stat().st_size)
with open(net) as f:
    print("Header:", f.readline().strip())

Wrote: /Users/mkrishan/Documents/boolearn-main/results/02_autoencoder_auto16/nets/auto16.wth
4
16 16 4 16 16

Running: /Users/mkrishan/Documents/boolearn-main/boolearn/src/layered 0.5 /Users/mkrishan/Documents/boolearn-main/results/02_autoencoder_auto16/nets/auto16.wth /Users/mkrishan/Documents/boolearn-main/results/02_autoencoder_auto16/nets/auto16.net
✅ net: /Users/mkrishan/Documents/boolearn-main/results/02_autoencoder_auto16/nets/auto16.net | bytes: 17316
Header: 69  17  16  692


In [9]:
import subprocess, time, re, select

run_id = "auto16_demo"
gap_path = RUN_DIR / f"{run_id}.gap"

items  = "16"
btfn   = "7.0"     # good default for autoencoder demo
beta   = "0.2"
gam    = "1e-4"
epochs = "80"
itermax= "200000"
gapstop= "0.05"
trials = "1"

cmd = [str(train), str(net), str(dat),
       items, btfn, beta, gam, epochs, itermax, gapstop, trials, run_id]

print("Running:", " ".join(cmd))
print("Watching:", gap_path.name)

proc = subprocess.Popen(cmd, cwd=str(RUN_DIR),
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)

last = 0
printed = False

try:
    while True:
        if proc.stdout is not None:
            r,_,_ = select.select([proc.stdout], [], [], 0.0)
            if r:
                _ = proc.stdout.readline()

        if gap_path.exists():
            lines = [ln.strip() for ln in gap_path.read_text().splitlines() if ln.strip()]
            if lines and not printed:
                print("\niter\ttrain%\ttest%")
                printed = True
            if len(lines) > last:
                for ln in lines[last:]:
                    it = ln.split()[0]
                    pcts = re.findall(r"(\d+(?:\.\d+)?)\s*%", ln)
                    if len(pcts) >= 2:
                        tr, te = float(pcts[-2]), float(pcts[-1])
                        print(f"{it}\t{tr:.3f}\t{te:.3f}")
                last = len(lines)

        if proc.poll() is not None:
            break
        time.sleep(0.2)

finally:
    try: proc.kill()
    except Exception: pass

print("\nReturn code:", proc.returncode)

print("\nArtifacts:")
for p in sorted(RUN_DIR.glob(run_id + ".*")):
    print(" -", p.name, "|", p.stat().st_size, "bytes")

Running: /Users/mkrishan/Documents/boolearn-main/boolearn/src/train /Users/mkrishan/Documents/boolearn-main/results/02_autoencoder_auto16/nets/auto16.net /Users/mkrishan/Documents/boolearn-main/data/autocode/auto16.dat 16 7.0 0.2 1e-4 80 200000 0.05 1 auto16_demo
Watching: auto16_demo.gap

iter	train%	test%
2	53.516	0.000
3	47.266	0.000
4	49.219	0.000
5	49.609	0.000
6	49.609	0.000
7	49.219	0.000
8	49.219	0.000
9	49.219	0.000
10	50.000	0.000
11	51.172	0.000
12	49.609	0.000
13	50.781	0.000
14	50.391	0.000
15	50.781	0.000
16	51.172	0.000
17	51.562	0.000
18	51.172	0.000
19	51.562	0.000
20	52.344	0.000
22	51.562	0.000
25	53.125	0.000
29	49.219	0.000
34	50.000	0.000
39	52.734	0.000
46	55.469	0.000
53	52.734	0.000
62	55.469	0.000
72	57.812	0.000
84	61.719	0.000
98	62.500	0.000
114	58.984	0.000
132	61.328	0.000
154	64.844	0.000
180	67.188	0.000
209	67.969	0.000
243	74.219	0.000
283	73.047	0.000
330	80.859	0.000
384	81.250	0.000
448	81.641	0.000
521	80.859	0.000
607	84.375	0.000
707	81.641	0.00

In [11]:
print("\n=== tail gap ===")
if gap_path.exists():
    print("\n".join(gap_path.read_text().splitlines()[-12:]))
else:
    print("No gap file found.")


=== tail gap ===
      3251    0.01285096    0.02894790    0.04947003    0.12854299    0.06358261    92.969 %     0.000 %
      3787    0.01407532    0.02595224    0.08491847    0.14511770    0.06358261    88.672 %     0.000 %
      4411    0.01804807    0.03176680    0.07452987    0.15830085    0.06358261    94.922 %     0.000 %
      5138    0.01005307    0.01954236    0.03091154    0.08739714    0.05886694    94.531 %     0.000 %
      5984    0.00963460    0.01754417    0.03277145    0.07894429    0.05886694    94.531 %     0.000 %
      6971    0.01551722    0.02897335    0.05083345    0.13311748    0.05886694    90.234 %     0.000 %
      8120    0.01248703    0.02226654    0.03979844    0.10344301    0.05886694    91.016 %     0.000 %
      9458    0.01450956    0.02962264    0.06606441    0.14054473    0.05763192    92.578 %     0.000 %
     11017    0.01911724    0.03612857    0.08724722    0.17760415    0.05763192    91.406 %     0.000 %
     12833    0.01832454    0.0308941